# 🏠 Boston Housing Dataset — Exploratory Data Analysis
> **Author:** Karan Singh &nbsp;|&nbsp; **Topic:** EDA & Data Storytelling &nbsp;|&nbsp; **Tools:** Python · Pandas · Seaborn · Matplotlib

---

## 📌 What This Notebook Covers
This is a **complete Exploratory Data Analysis (EDA)** of the famous Boston Housing dataset. The goal is to uncover the key factors that drive housing prices — before any modelling.

| Section | Focus |
|---|---|
| 1️⃣ Dataset Overview | Shape, columns, data types |
| 2️⃣ Missing Value Analysis | Null checks, heatmap |
| 3️⃣ Univariate Analysis | Distribution of every feature |
| 4️⃣ Target Variable Analysis | Price distribution & skewness |
| 5️⃣ Bivariate Analysis | Feature vs. Price relationships |
| 6️⃣ Correlation & Heatmap | Which features matter most? |
| 7️⃣ Outlier Detection | Boxplots across all features |
| 8️⃣ Key Insights Summary | Final takeaways |

---

### 📦 Dataset Column Reference
| Column | Description |
|---|---|
| `crim` | Per capita crime rate by town |
| `zn` | % residential land zoned for large lots |
| `indus` | % non-retail business acres per town |
| `chas` | Charles River dummy (1 = bounds river) |
| `nox` | Nitric oxide concentration (pollution) |
| `rm` | Average rooms per dwelling |
| `age` | % owner-occupied units built before 1940 |
| `dis` | Weighted distance to employment centres |
| `rad` | Index of highway accessibility |
| `tax` | Property-tax rate per $10,000 |
| `ptratio` | Pupil-teacher ratio by town |
| `b` | 1000(Bk−0.63)² — proxy for racial composition |
| `lstat` | % lower-status population |
| **`medv`** | **🎯 TARGET — Median home value ($1,000s)** |

---
## 🔧 Section 0 — Import Libraries

In [ ]:
# ── Core Libraries ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Visualisation ───────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Global Plot Style ───────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi'     : 120,
    'axes.titlesize' : 14,
    'axes.titleweight': 'bold',
    'axes.labelsize' : 12,
})

# ── Colour Palette (consistent across all plots) ────────────────
PRIMARY   = '#2563EB'   # Blue
SECONDARY = '#DC2626'   # Red
ACCENT    = '#16A34A'   # Green
WARM      = '#D97706'   # Amber
PURPLE    = '#7C3AED'   # Purple

print('✅ All libraries loaded successfully!')

---
## 1️⃣ Section 1 — Dataset Overview

In [ ]:
# ── Load Dataset ─────────────────────────────────────────────────
# If using Google Colab: upload the file and use the filename directly
# from google.colab import files; files.upload()

df = pd.read_csv('BostonHousing.csv')

print('=' * 50)
print(f'  📊 Dataset Shape : {df.shape[0]} rows × {df.shape[1]} columns')
print('=' * 50)
df.head()

In [ ]:
# ── Data Types & Non-Null Counts ─────────────────────────────────
# All columns should be numeric; chas is a binary dummy variable
df.info()

In [ ]:
# ── Statistical Summary ──────────────────────────────────────────
# KEY OBSERVATIONS FROM DESCRIBE:
# • crim has huge range (0.006 → 88.9) — extreme right-skew expected
# • rm (rooms) ranges 3.5 → 8.8 — normal-ish distribution expected
# • medv (target) ranges $5K → $50K with a mean of ~$22.5K
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

---
## 2️⃣ Section 2 — Missing Value Analysis

In [ ]:
# ── Missing Value Check ──────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print('Missing Values Summary:')
print(missing_df[missing_df['Missing Count'] > 0] if missing.sum() > 0 else '✅ No missing values found!')

# ── Missing Value Heatmap ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 2))
sns.heatmap(df.isnull().T, cbar=False, cmap='viridis', ax=ax, yticklabels=True)
ax.set_title('Missing Value Heatmap — Yellow = Missing, Purple = Present', pad=12)
ax.set_xlabel('Row Index')
plt.tight_layout()
plt.show()

# 💡 INSIGHT: The Boston Housing dataset is clean with zero missing values.
# No imputation required — we can proceed directly to analysis.

---
## 3️⃣ Section 3 — Univariate Analysis
> Examining the distribution of every individual feature to understand shape, spread, and outliers.

In [ ]:
# ── Distribution Plots for All Features ─────────────────────────
# A histogram + KDE for every column tells us:
#   • Is the feature normally distributed or skewed?
#   • Are there outliers pulling the tail?
#   • What is the most common value range?

cols  = df.columns.tolist()
n_col = 3
n_row = (len(cols) + n_col - 1) // n_col

fig, axes = plt.subplots(n_row, n_col, figsize=(18, n_row * 4))
axes = axes.flatten()

colors = [PRIMARY, SECONDARY, ACCENT, WARM, PURPLE,
          '#0891B2', '#BE185D', '#65A30D', '#B45309', '#6D28D9',
          '#0F766E', '#C2410C', '#1D4ED8', '#047857']

for i, col in enumerate(cols):
    skew_val = df[col].skew()
    sns.histplot(df[col], kde=True, ax=axes[i], color=colors[i % len(colors)],
                 alpha=0.6, edgecolor='white', linewidth=0.5)
    axes[i].set_title(f'{col.upper()}  |  Skew: {skew_val:.2f}')
    axes[i].set_xlabel('')
    # Annotate mean and median lines
    axes[i].axvline(df[col].mean(),   color='red',   linestyle='--', linewidth=1.5, label='Mean')
    axes[i].axvline(df[col].median(), color='black', linestyle='-',  linewidth=1.5, label='Median')
    axes[i].legend(fontsize=8)

# Hide any unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('📊 Univariate Distribution of All Features\n(Red Dashed = Mean | Black = Median)',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# 💡 INSIGHTS:
# • CRIM is heavily right-skewed — most towns have very low crime,
#   but a few towns have extremely high crime rates.
# • ZN (zoning) is zero for most towns — most land is mixed-use.
# • RM (rooms) is the most normally distributed feature.
# • MEDV shows slight right-skew with a cap at 50 (data censoring).

In [ ]:
# ── Skewness Summary Table ───────────────────────────────────────
# Skewness > +1 → Right skewed (long right tail)
# Skewness < -1 → Left skewed  (long left tail)
# Skewness ≈  0 → Roughly normal

skew_df = pd.DataFrame({
    'Feature'  : df.columns,
    'Skewness' : df.skew().values,
    'Type'     : ['Right-Skewed' if s > 0.5 else 'Left-Skewed' if s < -0.5 else 'Normal'
                  for s in df.skew().values]
}).sort_values('Skewness', ascending=False).reset_index(drop=True)

print('📐 Skewness Summary:')
skew_df.style.bar(subset=['Skewness'], color=['#DC2626', '#2563EB'])

---
## 4️⃣ Section 4 — Target Variable Deep Dive (MEDV)
> Understanding our prediction target — Median Home Value

In [ ]:
# ── Price Distribution — Raw vs Log-Transformed ──────────────────
# Log transformation is applied to reduce skewness.
# A more symmetric distribution often leads to better model performance.

prices  = df['medv']
y_log   = np.log(prices)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Raw Price
sns.histplot(prices, kde=True, ax=axes[0], color=PRIMARY, alpha=0.7, edgecolor='white')
axes[0].axvline(prices.mean(),   color='red',   linestyle='--', linewidth=2, label=f'Mean  = ${prices.mean():.1f}K')
axes[0].axvline(prices.median(), color='black', linestyle='-',  linewidth=2, label=f'Median= ${prices.median():.1f}K')
axes[0].set_title(f'Raw MEDV — Skewness: {prices.skew():.2f}')
axes[0].set_xlabel('Median Home Value ($1,000s)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Log Price
sns.histplot(y_log, kde=True, ax=axes[1], color=ACCENT, alpha=0.7, edgecolor='white')
axes[1].axvline(y_log.mean(),   color='red',   linestyle='--', linewidth=2, label=f'Mean   = {y_log.mean():.2f}')
axes[1].axvline(y_log.median(), color='black', linestyle='-',  linewidth=2, label=f'Median = {y_log.median():.2f}')
axes[1].set_title(f'Log(MEDV) — Skewness: {y_log.skew():.2f}  ✅ More Normal')
axes[1].set_xlabel('Log of Median Home Value')
axes[1].set_ylabel('Count')
axes[1].legend()

fig.suptitle('🏷️ Target Variable: Raw Price vs Log-Transformed Price',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# 💡 INSIGHT: The raw MEDV has a right-skew of ~1.1, with a noticeable
# spike at $50K — this is data censoring (all homes over $50K were
# capped at $50K in the original dataset). Log transformation reduces
# skewness from 1.1 → 0.08, making it nearly symmetric.

In [ ]:
# ── Price Statistics Summary ─────────────────────────────────────
print('🏠 Boston Housing Price Statistics')
print('=' * 40)
print(f'  Minimum Price  : ${prices.min():>7.1f}K')
print(f'  25th Percentile: ${prices.quantile(0.25):>7.1f}K')
print(f'  Median Price   : ${prices.median():>7.1f}K')
print(f'  Mean Price     : ${prices.mean():>7.2f}K')
print(f'  75th Percentile: ${prices.quantile(0.75):>7.1f}K')
print(f'  Maximum Price  : ${prices.max():>7.1f}K')
print(f'  Std Deviation  : ${prices.std():>7.2f}K')
print(f'  Skewness       : {prices.skew():>8.3f}')
print('=' * 40)

# ── Price Range Segmentation ─────────────────────────────────────
bins   = [0, 15, 25, 35, 50]
labels = ['Budget (<$15K)', 'Mid-Range ($15–25K)', 'Premium ($25–35K)', 'Luxury ($35K+)']
df['price_segment'] = pd.cut(df['medv'], bins=bins, labels=labels)

fig, ax = plt.subplots(figsize=(10, 5))
segment_counts = df['price_segment'].value_counts().sort_index()
bars = ax.bar(segment_counts.index, segment_counts.values,
               color=[SECONDARY, PRIMARY, ACCENT, WARN := WARM], edgecolor='white', linewidth=1.5)

for bar in bars:
    pct = bar.get_height() / len(df) * 100
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f'{bar.get_height()}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('🏘️ Distribution of Homes by Price Segment')
ax.set_xlabel('Price Segment')
ax.set_ylabel('Number of Homes')
ax.set_ylim(0, segment_counts.max() + 40)
plt.tight_layout()
plt.show()

df.drop(columns='price_segment', inplace=True)  # clean up

# 💡 INSIGHT: Nearly half of all homes fall in the Mid-Range $15–25K bracket.
# Only ~19% of homes qualify as Luxury (>$35K), with the spike at $50K
# confirming the dataset's censoring effect at the top end.

---
## 5️⃣ Section 5 — Bivariate Analysis
> How does each feature individually relate to house price?

In [ ]:
# ── Scatter Plots: Every Feature vs MEDV ─────────────────────────
# Shows linear/non-linear relationships, outliers, and direction of effect.

feature_cols = [c for c in df.columns if c != 'medv']
n_col = 3
n_row = (len(feature_cols) + n_col - 1) // n_col

fig, axes = plt.subplots(n_row, n_col, figsize=(18, n_row * 4.5))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    corr = df[col].corr(df['medv'])
    color = PRIMARY if corr > 0 else SECONDARY
    axes[i].scatter(df[col], df['medv'], alpha=0.35, s=25, color=color, edgecolors='none')
    # Add a trend line
    m, b = np.polyfit(df[col], df['medv'], 1)
    x_line = np.linspace(df[col].min(), df[col].max(), 100)
    axes[i].plot(x_line, m * x_line + b, color='black', linewidth=2, linestyle='--')
    axes[i].set_title(f'{col.upper()} vs MEDV  |  r = {corr:.2f}')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('MEDV ($1,000s)')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('🔍 Bivariate Analysis: Each Feature vs House Price\n(Blue = Positive Correlation | Red = Negative)',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# 💡 KEY INSIGHTS:
# • RM (rooms) shows the STRONGEST positive correlation (r ≈ +0.70)
# • LSTAT (% lower status) shows the STRONGEST negative correlation (r ≈ -0.74)
# • PTRATIO (pupil-teacher ratio) has a moderate negative relationship
# • CRIM, NOX show non-linear (curved) negative relationships

In [ ]:
# ── Top 2 Most Impactful Features — Deep Dive ────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# RM vs MEDV
sns.regplot(x='rm', y='medv', data=df, ax=axes[0],
            scatter_kws={'alpha': 0.5, 's': 50, 'color': PRIMARY},
            line_kws={'color': 'black', 'linewidth': 2})
corr_rm = df['rm'].corr(df['medv'])
axes[0].set_title(f'🛏️ Avg. Rooms (RM) vs Price  |  r = {corr_rm:.3f}')
axes[0].set_xlabel('Average Number of Rooms')
axes[0].set_ylabel('Median Home Value ($1,000s)')
axes[0].annotate('More rooms → Higher price\nStrong positive relationship',
                  xy=(0.05, 0.88), xycoords='axes fraction',
                  fontsize=10, color=PRIMARY,
                  bbox=dict(boxstyle='round,pad=0.3', facecolor='#EFF6FF', edgecolor=PRIMARY))

# LSTAT vs MEDV
sns.regplot(x='lstat', y='medv', data=df, ax=axes[1],
            scatter_kws={'alpha': 0.5, 's': 50, 'color': SECONDARY},
            line_kws={'color': 'black', 'linewidth': 2})
corr_lstat = df['lstat'].corr(df['medv'])
axes[1].set_title(f'👥 Lower Status % (LSTAT) vs Price  |  r = {corr_lstat:.3f}')
axes[1].set_xlabel('% Lower-Status Population')
axes[1].set_ylabel('Median Home Value ($1,000s)')
axes[1].annotate('Higher poverty % → Lower price\nStrongest negative relationship',
                  xy=(0.35, 0.88), xycoords='axes fraction',
                  fontsize=10, color=SECONDARY,
                  bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFF1F2', edgecolor=SECONDARY))

fig.suptitle('🎯 Two Most Powerful Predictors of House Price',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── CHAS (Charles River) — Categorical Bivariate Analysis ────────
# Does living near the Charles River significantly affect home price?

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
sns.boxplot(x='chas', y='medv', data=df, ax=axes[0],
            palette={0: PRIMARY, 1: ACCENT},
            width=0.5, linewidth=1.5)
axes[0].set_xticklabels(['Not Near River\n(chas=0)', 'Near River\n(chas=1)'])
axes[0].set_title('Charles River Proximity vs Home Price')
axes[0].set_ylabel('Median Home Value ($1,000s)')
axes[0].set_xlabel('')

# Violin
sns.violinplot(x='chas', y='medv', data=df, ax=axes[1],
               palette={0: PRIMARY, 1: ACCENT},
               inner='quartile', linewidth=1.5)
axes[1].set_xticklabels(['Not Near River', 'Near River'])
axes[1].set_title('Price Distribution Shape — Violin Plot')
axes[1].set_ylabel('Median Home Value ($1,000s)')
axes[1].set_xlabel('')

# Annotate median prices
for chas_val, ax in zip([0, 1], [axes[0], axes[1]]):
    med = df[df['chas'] == chas_val]['medv'].median()

chas_0_med = df[df['chas'] == 0]['medv'].median()
chas_1_med = df[df['chas'] == 1]['medv'].median()
print(f'💧 Median Price — Not Near River : ${chas_0_med:.1f}K')
print(f'💧 Median Price — Near River     : ${chas_1_med:.1f}K')
print(f'📈 River Premium                 : +${chas_1_med - chas_0_med:.1f}K ({(chas_1_med/chas_0_med - 1)*100:.1f}% higher)')

fig.suptitle('🌊 Does Living Near the Charles River Increase Home Value?',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 💡 INSIGHT: Homes near the Charles River command a significant price premium.
# The median price is higher for river-adjacent homes, confirming that
# waterfront proximity adds measurable value — a classic real-estate principle.

In [ ]:
# ── Highway Accessibility (RAD) — Bar Analysis ───────────────────
# RAD is a categorical-like index (1–24) measuring highway accessibility

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Count by RAD index
rad_counts = df['rad'].value_counts().sort_index()
axes[0].bar(rad_counts.index.astype(str), rad_counts.values,
            color=PURPLE, edgecolor='white', linewidth=1)
axes[0].set_title('Number of Towns by Highway Access Index (RAD)')
axes[0].set_xlabel('RAD Index')
axes[0].set_ylabel('Number of Towns')

# Average price by RAD
rad_price = df.groupby('rad')['medv'].median().reset_index()
axes[1].bar(rad_price['rad'].astype(str), rad_price['medv'],
            color=WARM, edgecolor='white', linewidth=1)
axes[1].set_title('Median Home Price by Highway Access Index (RAD)')
axes[1].set_xlabel('RAD Index')
axes[1].set_ylabel('Median Price ($1,000s)')

fig.suptitle('🛣️ Highway Accessibility (RAD) — Count & Price Impact',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 💡 INSIGHT: RAD index of 24 dominates (most common) and corresponds to
# towns with very high highway access — but these also tend to have
# lower prices due to noise, pollution, and industrial character.

---
## 6️⃣ Section 6 — Correlation Analysis
> Which features are strongly related to each other and to price?

In [ ]:
# ── Full Correlation Heatmap ──────────────────────────────────────
# Correlation ranges: -1 (perfect negative) to +1 (perfect positive)
# Values close to 0 indicate little linear relationship

corr_matrix = df.corr(numeric_only=True)

# Mask upper triangle for cleaner look
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(corr_matrix,
            mask=mask,
            annot=True, fmt='.2f',
            cmap='RdBu_r',
            center=0,
            vmin=-1, vmax=1,
            annot_kws={'size': 10},
            linewidths=0.5,
            linecolor='white',
            square=True,
            ax=ax)

ax.set_title('🔥 Correlation Heatmap — Boston Housing Features\n(Red = Positive | Blue = Negative)',
             fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# ── Feature Correlation with MEDV (Sorted Bar Chart) ─────────────
# This ranks every feature by its absolute correlation with house price

medv_corr = corr_matrix['medv'].drop('medv').sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
colors_bar = [SECONDARY if v < 0 else ACCENT for v in medv_corr.values]
bars = ax.barh(medv_corr.index, medv_corr.values,
               color=colors_bar, edgecolor='white', linewidth=1)

for bar, val in zip(bars, medv_corr.values):
    ax.text(val + (0.01 if val >= 0 else -0.01),
            bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center',
            ha='left' if val >= 0 else 'right',
            fontsize=10, fontweight='bold')

ax.axvline(0, color='black', linewidth=1)
ax.set_title('📊 Feature Correlation with House Price (MEDV)\nGreen = Positive Effect | Red = Negative Effect',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Pearson Correlation Coefficient')
ax.set_xlim(-0.85, 0.85)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

# 💡 KEY FINDINGS:
# POSITIVE DRIVERS (increase price):
#   • RM (+0.70)  — More rooms = higher value. Strongest positive signal.
#   • ZN (+0.36)  — Residential zoning correlates with better neighbourhoods.
#   • B  (+0.33)  — Weak positive signal.
#   • DIS (+0.25) — Farther from employment = slightly suburban premium.
#
# NEGATIVE DRIVERS (decrease price):
#   • LSTAT (-0.74) — % poverty is the single strongest price depressor.
#   • PTRATIO (-0.51) — More students per teacher = worse school quality.
#   • INDUS (-0.48)  — Industrial zones lower residential desirability.
#   • NOX (-0.43)    — Pollution significantly lowers property values.
#   • CRIM (-0.39)   — Crime rate reduces desirability and value.

In [ ]:
# ── Notable Feature Pair: NOX vs DIS ─────────────────────────────
# Pollution DECREASES as distance from employment INCREASES
# This reveals the industrial geography of Boston

dis_nox = df['nox'].corr(df['dis'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Scatter
sns.scatterplot(data=df, x='dis', y='nox', alpha=0.5, s=60, color=PURPLE, ax=axes[0])
axes[0].set_title(f'NOX vs DIS  |  r = {dis_nox:.3f}')
axes[0].set_xlabel('DIS — Distance from Employment Centres')
axes[0].set_ylabel('NOX — Nitric Oxide Concentration')
axes[0].annotate('Closer to jobs = More pollution\n(industrial zones near employment)',
                  xy=(0.35, 0.88), xycoords='axes fraction',
                  fontsize=10, color=PURPLE,
                  bbox=dict(boxstyle='round', facecolor='#F5F3FF', edgecolor=PURPLE))

# Joint KDE
axes[1].hexbin(df['dis'], df['nox'], gridsize=25, cmap='Purples', mincnt=1)
axes[1].set_title('Density Hexplot — DIS vs NOX')
axes[1].set_xlabel('DIS — Distance from Employment')
axes[1].set_ylabel('NOX — Nitric Oxide')
plt.colorbar(axes[1].collections[0], ax=axes[1], label='Count')

fig.suptitle('🏭 Pollution vs Distance from Employment — Spatial Insight',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 💡 INSIGHT: Strong negative correlation (r = -0.77) between NOX and DIS.
# Towns closest to employment centres have the highest pollution.
# This reflects the historical industrial geography of Boston —
# factories and workplaces were pollution sources.

In [ ]:
# ── Multicollinearity Check: TAX vs RAD ──────────────────────────
# These two features are highly correlated (r ≈ 0.91) — potential multicollinearity
# which causes issues in regression models

tax_rad = df['tax'].corr(df['rad'])

fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=df, x='rad', y='tax', alpha=0.6, s=80, color=WARM, ax=ax)
ax.set_title(f'⚠️ TAX vs RAD — High Multicollinearity Risk  |  r = {tax_rad:.3f}')
ax.set_xlabel('RAD — Highway Accessibility Index')
ax.set_ylabel('TAX — Property Tax Rate per $10,000')
ax.annotate(f'r = {tax_rad:.3f}\nThese two features carry\nredundant information!',
             xy=(0.05, 0.78), xycoords='axes fraction',
             fontsize=11, color=SECONDARY,
             bbox=dict(boxstyle='round', facecolor='#FFF7ED', edgecolor=WARM))
plt.tight_layout()
plt.show()

# 💡 INSIGHT: TAX and RAD are nearly perfectly correlated (r = 0.91+).
# Towns with high highway access tend to also have high property tax rates.
# This multicollinearity inflates VIF scores and can distort regression coefficients.
# In model-building, one of these should be dropped or regularization applied.

---
## 7️⃣ Section 7 — Outlier Detection
> Identifying extreme values that could influence model training

In [ ]:
# ── Boxplots — All Features ───────────────────────────────────────
# Whiskers extend to 1.5×IQR; dots beyond = statistical outliers

fig, axes = plt.subplots(3, 5, figsize=(20, 12))
axes = axes.flatten()

for i, col in enumerate(df.select_dtypes(include='number').columns):
    q1  = df[col].quantile(0.25)
    q3  = df[col].quantile(0.75)
    iqr = q3 - q1
    n_outliers = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()

    bp = axes[i].boxplot(df[col].dropna(), patch_artist=True,
                          medianprops={'color': 'black', 'linewidth': 2},
                          flierprops={'marker': 'o', 'markersize': 4, 'alpha': 0.5,
                                      'markerfacecolor': SECONDARY})
    bp['boxes'][0].set_facecolor(f'{PRIMARY}40')
    axes[i].set_title(f'{col.upper()}\n{n_outliers} outliers', fontsize=11)
    axes[i].set_xticks([])

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('📦 Outlier Detection — Boxplots for All Features\n(Red dots = Statistical Outliers beyond 1.5×IQR)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Outlier Summary Table ─────────────────────────────────────────
# Count outliers per feature and assess severity

outlier_summary = []
for col in df.select_dtypes(include='number').columns:
    q1  = df[col].quantile(0.25)
    q3  = df[col].quantile(0.75)
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
    pct   = n_out / len(df) * 100
    outlier_summary.append({
        'Feature'       : col,
        'Outlier Count' : n_out,
        'Outlier %'     : round(pct, 2),
        'Severity'      : '🔴 High' if pct > 10 else ('🟡 Medium' if pct > 5 else '🟢 Low')
    })

out_df = pd.DataFrame(outlier_summary).sort_values('Outlier %', ascending=False).reset_index(drop=True)
print('🚨 Outlier Summary:')
out_df

# 💡 INSIGHT:
# CRIM has the most severe outlier problem — a handful of towns
# have extraordinarily high crime rates (~88 vs typical <1).
# B (racial composition proxy) also has high outlier concentration.
# These outliers should be handled before regression modeling.

---
## 8️⃣ Section 8 — Pairplot (Select Key Features)
> Pairwise relationships between the most important features

In [ ]:
# ── Selective Pairplot — Top Features Only ────────────────────────
# Full pairplot (14 features) is hard to read.
# We focus on the 6 most correlated features with MEDV.

top_features = ['medv', 'lstat', 'rm', 'ptratio', 'nox', 'crim', 'dis']

pair_df = df[top_features].copy()

# Colour code by price quartile for visual richness
pair_df['Price Tier'] = pd.qcut(df['medv'], q=3, labels=['Low', 'Mid', 'High'])

g = sns.pairplot(pair_df, hue='Price Tier',
                  palette={'Low': SECONDARY, 'Mid': WARM, 'High': ACCENT},
                  diag_kind='kde', plot_kws={'alpha': 0.4, 's': 20},
                  corner=True)

g.figure.suptitle('🔗 Pairplot — Top 6 Features Coloured by Price Tier',
                   y=1.01, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 💡 INSIGHT: High-price homes (green) cluster clearly:
# • At HIGH RM values (many rooms)
# • At LOW LSTAT values (wealthier neighbourhoods)
# • At LOW CRIM values (safe areas)
# This visual confirms what correlation numbers told us.

---
## ✅ Section 9 — EDA Summary & Key Insights
> Everything we learned before touching a single model

In [ ]:
# ── Final Summary Dashboard ───────────────────────────────────────
# A concise visual summary of all key findings from EDA

fig = plt.figure(figsize=(18, 10))
fig.patch.set_facecolor('#F8FAFC')

# Panel 1: Correlation Ranking
ax1 = fig.add_subplot(1, 3, 1)
corr_vals = corr_matrix['medv'].drop('medv').sort_values()
colors_final = [SECONDARY if v < 0 else ACCENT for v in corr_vals.values]
ax1.barh(corr_vals.index, corr_vals.values, color=colors_final, edgecolor='white')
ax1.axvline(0, color='black', linewidth=0.8)
ax1.set_title('Feature → Price Correlation', fontweight='bold')
ax1.set_xlabel('r value')
ax1.grid(axis='x', alpha=0.3)

# Panel 2: Price Distribution
ax2 = fig.add_subplot(1, 3, 2)
sns.histplot(df['medv'], kde=True, ax=ax2, color=PRIMARY, alpha=0.7)
ax2.axvline(df['medv'].mean(),   color='red',   linestyle='--', linewidth=2, label=f"Mean=${df['medv'].mean():.1f}K")
ax2.axvline(df['medv'].median(), color='black', linestyle='-',  linewidth=2, label=f"Median=${df['medv'].median():.1f}K")
ax2.set_title('House Price Distribution', fontweight='bold')
ax2.set_xlabel('MEDV ($1,000s)')
ax2.legend(fontsize=9)

# Panel 3: RM vs MEDV
ax3 = fig.add_subplot(1, 3, 3)
ax3.scatter(df['rm'], df['medv'], alpha=0.4, s=30, color=PURPLE)
m, b = np.polyfit(df['rm'], df['medv'], 1)
x_r  = np.linspace(df['rm'].min(), df['rm'].max(), 100)
ax3.plot(x_r, m * x_r + b, color='black', linewidth=2)
ax3.set_title(f'RM vs MEDV  (r = {df["rm"].corr(df["medv"]):.2f})', fontweight='bold')
ax3.set_xlabel('Average Rooms')
ax3.set_ylabel('Price ($1,000s)')

fig.suptitle('🏠 Boston Housing EDA — Summary Dashboard',
             fontsize=17, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 📝 Final EDA Conclusions

### 🔑 What This EDA Reveals

| Finding | Detail |
|---|---|
| **Dataset** | 506 rows × 14 columns, zero missing values, clean and ready |
| **Target Skew** | MEDV is right-skewed (1.1) — log transformation recommended for regression |
| **Strongest Predictor (+)** | `RM` (+0.70) — Every additional room adds significant value |
| **Strongest Predictor (−)** | `LSTAT` (−0.74) — Socioeconomic status is the top price depressor |
| **River Premium** | Homes near Charles River are **~$4K more expensive** on average |
| **Multicollinearity** | `TAX` & `RAD` are correlated at r ≈ 0.91 — drop one before modelling |
| **Pollution Effect** | `NOX` strongly correlates with employment proximity — industrial hubs pollute |
| **Top Outlier** | `CRIM` — Most towns safe, but few extreme outliers pull the mean |
| **Best 4 Features** | `lstat`, `rm`, `ptratio`, `nox` explain most of house price variance |

### 🚀 Recommendations for Modelling
1. **Log-transform MEDV** before running regression — reduces skewness from 1.1 → 0.08
2. **Drop `indus` and `age`** — statistically insignificant (high p-values)
3. **Handle `TAX`/`RAD` multicollinearity** — keep only one or use regularization
4. **Scale features** using StandardScaler before any distance-based models
5. **Investigate CRIM outliers** — consider capping or log-transforming

---
*📌 This EDA was built as part of a Machine Learning learning journey.*  
*Dataset: Boston Housing | Tools: Python, Pandas, Seaborn, Matplotlib*